# Batch export SECORE H10 IBI/RMSSD to NCDF

This notebook builds `h10_xarray` with `build_h10_ibi_rmssd_xarray_auto` and exports SECORE NCDF files for all dyads from the same list used in EEG batch export.

In [6]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

# Add project root so src package imports work from scripts/
sys.path.insert(0, os.path.abspath('..'))

from src.secore_loader import build_h10_ibi_rmssd_xarray_auto

In [7]:
input_folder = "/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_RAW_DATA"
export_folder = "/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_EEG_exported"
input_synchronization_data = os.path.join(input_folder, "timings_secore_hrv.csv")
# Hardcoded H10 device IDs (same as secore_raw_import_demo.ipynb)
DEV_CG = "A839C92B"  # Caregiver H10 device ID
DEV_CH = "A83E1E24"  # Child H10 device ID

# Build settings
plot_flag = False
fs_ibi = 8
window_size_rmssd_s = 30
decimate_factor_loader = 8
decimate_factor_align = 16
selected_time = (0, 220)

In [ ]:
# Informacja od Weroniki:
# Diady które na pewno można wykluczyć z takiego automatycznego eksportu, bo nie miały eeg lub h10 wg Warsaw IDs, to:
# 6,7,8,10,11,33,46,47,79,90,93,96 (na czerwono nie mają obu modalności)
# Dodatkowo diada 3 ma h10 tylko od dziecka, a 92 ma eeg zapisane w dwóch plikach (wywaliło się nagrywanie w trakcie).
not_complete_dyads = ['W_003','W_006','W_007','W_008','W_010','W_011',
                      'W_033','W_046','W_047','W_079','W_090','W_093','W_096']
                
# Same dyad list as in export_EEG_UNIWAW_dyades_to_ncdf.ipynb
dyades_to_export = [
    'W_000', 'W_016',          'W_049', 'W_086',
    'W_001', 'W_017', 'W_034', 'W_053', 'W_087',
    'W_002', 'W_018', 'W_035', 'W_057', 'W_088',
             'W_019', 'W_036', 'W_064', 
    'W_004', 'W_020', 'W_037', 'W_071', 'W_091',
    'W_005', 'W_021', 'W_038', 'W_072', 'W_092',
             'W_022', 'W_039', 'W_073', 
    'W_024', 'W_040', 'W_074', 'W_094',
             'W_025', 'W_041', 'W_075', 
    'W_009', 'W_026', 'W_042', 'W_076', 'W_097',
             'W_027', 'W_043', 'W_077', 'W_104',
             'W_028', 'W_044', 'W_078', 'W_112',
    'W_012', 'W_029', 'W_045',        
    'W_013', 'W_030',          'W_081',
    'W_014', 'W_031',          'W_084',
    'W_015', 'W_032', 'W_048', 'W_085'
]

metadata_file = os.path.join(input_folder, "meta_data.csv")
metadata_df = pd.read_csv(metadata_file, sep=';')
synchronization_df = pd.read_csv(input_synchronization_data, sep=',')

def sec_ms_str_to_float(value):
    if pd.isna(value):
        return np.nan

    s = str(value).strip().replace("\xa0", " ").replace("\u202f", " ")
    if not s:
        return np.nan

    parts = s.split()

    # Pattern like "18 199" or "1 191 372": groups of 3 digits after the first group.
    # Interpret as total milliseconds and convert to seconds.
    if len(parts) >= 2 and all(p.lstrip('-').isdigit() for p in parts):
        if all(len(p) == 3 for p in parts[1:]):
            sign = -1 if parts[0].startswith('-') else 1
            head = parts[0].lstrip('-')
            total_ms = int(head + ''.join(parts[1:]))
            return sign * (total_ms / 1000.0)

        # Legacy pattern: "sec ms" -> sec + ms/1000
        sec = float(parts[0].replace(',', '.'))
        ms = float(parts[1].replace(',', '.'))
        return sec + ms / 1000.0

    return float(s.replace(',', '.'))


dyades_to_export = [dyad for dyad in dyades_to_export if dyad in metadata_df['ID'].values]
dyades_to_export = sorted(dyades_to_export)
print(f"Exporting {len(dyades_to_export)} dyads: {dyades_to_export}")

Exporting 75 dyads: ['W_000', 'W_001', 'W_002', 'W_003', 'W_004', 'W_005', 'W_006', 'W_007', 'W_008', 'W_009', 'W_010', 'W_011', 'W_012', 'W_013', 'W_014', 'W_015', 'W_016', 'W_017', 'W_018', 'W_019', 'W_020', 'W_021', 'W_022', 'W_024', 'W_025', 'W_026', 'W_027', 'W_028', 'W_029', 'W_030', 'W_031', 'W_032', 'W_033', 'W_034', 'W_035', 'W_036', 'W_037', 'W_038', 'W_039', 'W_040', 'W_041', 'W_042', 'W_043', 'W_044', 'W_045', 'W_046', 'W_047', 'W_048', 'W_049', 'W_053', 'W_057', 'W_064', 'W_071', 'W_072', 'W_073', 'W_074', 'W_075', 'W_076', 'W_077', 'W_078', 'W_079', 'W_081', 'W_084', 'W_085', 'W_086', 'W_087', 'W_088', 'W_090', 'W_091', 'W_092', 'W_093', 'W_094', 'W_096', 'W_097', 'W_104']


In [9]:
def _sanitize_attrs_for_netcdf(attrs):
    out = {}
    for key, value in attrs.items():
        if value is None:
            out[key] = ""
        elif isinstance(value, (str, bytes, bool, int, float, np.number)):
            out[key] = value
        elif isinstance(value, (list, tuple, dict)):
            out[key] = json.dumps(value, ensure_ascii=False, default=str)
        else:
            out[key] = str(value)
    return out


def export_h10_to_secore_ncdf(h10_xarray, dyad_id, export_root):
    sampling_freq = float(h10_xarray.attrs.get("sampling_frequency_Hz", np.nan))

    event_windows = json.loads(h10_xarray.attrs.get("event_windows_s_json", "{}"))
    events_start_s = {name: float(win["start_s"]) for name, win in event_windows.items()}
    events_duration_s = {
        name: float(win["end_s"]) - float(win["start_s"]) for name, win in event_windows.items()
    }
    event_order = [k for k, _ in sorted(events_start_s.items(), key=lambda kv: kv[1])]

    metadata_payload = {
        "notes": "",
        "child_info": {},
        "event_order": event_order,
        "secore_event_windows_s": event_windows,
    }

    time_values = h10_xarray.coords["time"].values.astype(float)

    export_plan = [
        ("Secore_IBI", "IBI", "IBI_CH", "ch", "child", "IBI"),
        ("Secore_IBI", "IBI", "IBI_CG", "cg", "caregiver", "IBI"),
        ("Secore_RMSSD", "RMSSD", "RMSSD_CH", "ch", "child", "RMSSD"),
        ("Secore_RMSSD", "RMSSD", "RMSSD_CG", "cg", "caregiver", "RMSSD"),
    ]

    saved_files = []
    for export_folder_name, modality, src_channel, who, member_folder, out_channel in export_plan:
        if src_channel not in h10_xarray.channel.values:
            raise ValueError(f"Missing channel {src_channel} in h10_xarray for {dyad_id}.")

        sig_values = h10_xarray.sel(channel=src_channel).values.astype(float)
        signals = xr.DataArray(
            data=sig_values[:, None],
            coords={"time": time_values, "channel": [out_channel]},
            dims=["time", "channel"],
            name="signals",
        )

        attrs = {
            "dyad_id": dyad_id,
            "who": who,
            "sampling_freq": sampling_freq,
            "event_name": "Secore",
            "event_start": float(time_values[0]),
            "event_duration": float(time_values[-1] - time_values[0]),
            "time_margin_s": 0.0,
            "channel_names_csv": out_channel,
            "channel_names_json": json.dumps([out_channel], ensure_ascii=True),
            "events_start_s_json": events_start_s,
            "events_duration_s_json": events_duration_s,
            "metadata_json": metadata_payload,
        }
        signals.attrs.update(_sanitize_attrs_for_netcdf(attrs))

        out_dir = Path(export_root) / export_folder_name / dyad_id / member_folder
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = out_dir / f"{dyad_id}_{modality}_{who}_Secore.nc"

        signals.to_netcdf(out_file, engine="netcdf4", format="NETCDF4_CLASSIC")
        saved_files.append(str(out_file))

    return saved_files

In [10]:
failed_dyads = []
saved_files_all = []

for dyad in dyades_to_export:
    sync_row = synchronization_df.loc[synchronization_df['id'] == dyad]

    if sync_row.empty:
        print(f"Skipping {dyad}: dyad synchronization is yet unavailable")
        continue

    check_value = pd.to_numeric(sync_row['checked'], errors='coerce').iat[0]
    if check_value != 1:
        print(f"Skipping {dyad}: dyad synchronization is yet unavailable")
        continue

    video_timings = {
        "T1": float(sec_ms_str_to_float(sync_row['T1'].iat[0])),
        "T2": float(sec_ms_str_to_float(sync_row['T2'].iat[0])),
        "T3": float(sec_ms_str_to_float(sync_row['T3'].iat[0])),
        "T4": float(sec_ms_str_to_float(sync_row['T4'].iat[0])),
    }
    try:
        print(f"Building/exporting {dyad}...")

        dyad_nr = int(dyad.split('_')[1])
        synch_time = video_timings["T1"]
        h10_xarray = build_h10_ibi_rmssd_xarray_auto(
            dyad_nr=dyad_nr,
            video_timings=video_timings,
            data_base_path=input_folder,
            fs_ibi=fs_ibi,
            window_size_rmssd_s=window_size_rmssd_s,
            decimate_factor_loader=decimate_factor_loader,
            decimate_factor_align=decimate_factor_align,
            selected_time=selected_time,
            plot=plot_flag,
            preferred_dev_ch=DEV_CH,
            preferred_dev_cg=DEV_CG,
        )

        saved_files = export_h10_to_secore_ncdf(
            h10_xarray=h10_xarray,
            dyad_id=dyad,
            export_root=export_folder,
        )
        saved_files_all.extend(saved_files)

        print(f"Done: {dyad} ({len(saved_files)} files)")
    except Exception as e:
        failed_dyads.append((dyad, str(e)))
        print(f"Failed: {dyad} -> {e}")

print(
    f"Finished. Success: {len(dyades_to_export) - len(failed_dyads)}, "
    f"Failed: {len(failed_dyads)}, Exported files: {len(saved_files_all)}"
)

if failed_dyads:
    print("Failed dyads:")
    for dyad, err in failed_dyads:
        print(f"  - {dyad}: {err}")

Skipping W_000: dyad synchronization is yet unavailable
Building/exporting W_001...
Auto-detected latest recording for W_1: None None, CH=A83E1E24, CG=A839C92B
Detected events: {'Brave': {'name': 'Brave', 'start': 131.8779296875, 'duration': 60.1484375}, 'Peppa': {'name': 'Peppa', 'start': 272.556640625, 'duration': 60.46484375}, 'Incredibles': {'name': 'Incredibles', 'start': 202.259765625, 'duration': 60.046875}}
Applying iir filters to EEG data.
Reseting the EEG time to the start of Brave
No ET_event found, using EEG_events data only.
Events column created based on EEG_events and ET_event columns.
Event structure created based on events column.

Event Name                     Start (s)       Duration (s)   
Brave                          0.00            60.14          
Incredibles                    70.39           60.04          
Peppa                          140.68          60.45          

Computed lags (in seconds) to ECG: CG=2933.875, CH=2843.25
Done: W_001 (4 files)
Building/

/Users/admin/SYNCC-IN_local_FUW/hyperscanning-signal-analysis/src/secore_loader.py:27: UserWarning: genfromtxt: Empty input file: "/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_RAW_DATA/W_003/eeg/W_003_A839C92B_IBI.csv"
  data = np.genfromtxt(path, delimiter=",")


Detected events: {'Brave': {'name': 'Brave', 'start': 263.7421875, 'duration': 59.3310546875}, 'Peppa': {'name': 'Peppa', 'start': 333.306640625, 'duration': 59.6298828125}, 'Incredibles': {'name': 'Incredibles', 'start': 403.169921875, 'duration': 59.212890625}, 'Talk_1': {'name': 'Talk_1', 'start': 603.2099609375, 'duration': 181.0556640625}, 'Talk_2': {'name': 'Talk_2', 'start': 898.244140625, 'duration': 181.0546875}}
Applying iir filters to EEG data.
Reseting the EEG time to the start of Brave
No ET_event found, using EEG_events data only.
Events column created based on EEG_events and ET_event columns.
Event structure created based on events column.

Event Name                     Start (s)       Duration (s)   
Brave                          0.00            59.33          
Peppa                          69.57           59.62          
Incredibles                    139.43          59.21          
Talk_1                         339.47          181.05         
Talk_2               

/Users/admin/SYNCC-IN_local_FUW/hyperscanning-signal-analysis/.venv_SYNCCIN_local_FUW/lib/python3.12/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


Failed: W_006 -> integer division or modulo by zero
Building/exporting W_007...
Auto-detected latest recording for W_7: None None, CH=A83E1E24, CG=A839C92B
Detected events: {}
Applying iir filters to EEG data.
Failed: W_007 -> single positional indexer is out-of-bounds
Skipping W_008: dyad synchronization is yet unavailable
Building/exporting W_009...
Auto-detected latest recording for W_9: None None, CH=A83E1E24, CG=A839C92B
Detected events: {'Brave': {'name': 'Brave', 'start': 308.4150390625, 'duration': 59.298828125}, 'Peppa': {'name': 'Peppa', 'start': 447.4443359375, 'duration': 59.630859375}, 'Incredibles': {'name': 'Incredibles', 'start': 377.9462890625, 'duration': 59.21484375}, 'Talk_1': {'name': 'Talk_1', 'start': 633.20703125, 'duration': 181.0732421875}, 'Talk_2': {'name': 'Talk_2', 'start': 899.216796875, 'duration': 181.0400390625}}
Applying iir filters to EEG data.
Reseting the EEG time to the start of Brave
No ET_event found, using EEG_events data only.
Events column cr

In [11]:
# Preview first saved files
for p in saved_files_all[:10]:
    print(p)

/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_EEG_exported/Secore_IBI/W_001/child/W_001_IBI_ch_Secore.nc
/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_EEG_exported/Secore_IBI/W_001/caregiver/W_001_IBI_cg_Secore.nc
/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_EEG_exported/Secore_RMSSD/W_001/child/W_001_RMSSD_ch_Secore.nc
/Users/admin/Library/CloudStorage/GoogleDrive-j.zygierewicz@uw.edu.pl/.shortcut-targets-by-id/1N4ySQ5GO6UE8fY2jnRkRUjBFm4XHrBRv/SYNCC-IN/WP4          - Joint study/UniWAW Data collection/UNIWAW_EEG_exported/Secore_RMSSD/W_001/caregiv